# 🏆 Gold Layer: Air Quality (2023-2025)

**Dataset**: รวมข้อมูล Air Quality จาก 3 ปีเต็ม (2023-2025)

**Features**: 52 columns รวม temporal, lag, rolling, และ station aggregates

**Ready for**: ML model training, forecasting, analysis

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

GOLD_FILE = Path('data/gold/airquality_combined/airquality_2023_2025.parquet')

print("✅ Imports loaded")

## Load Gold Layer

In [ ]:
df = pl.read_parquet(GOLD_FILE)

print("=" * 100)
print(" " * 30 + "🏆 GOLD LAYER LOADED")
print("=" * 100)
print(f"\nTotal rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Date range: {df['timestamp_utc'].min()} → {df['timestamp_utc'].max()}")
print(f"Stations: {df['stationID'].n_unique()}")
print(f"\nMemory usage: {df.estimated_size('mb'):.1f} MB")
print("=" * 100)

## Schema Overview

In [ ]:
print("\n📋 Schema:")
print(df.schema)

print("\n🔢 Feature Categories:")
print(f"\nOriginal columns (16):")
original = ['stationID', 'lat', 'lon', 'timestamp_utc', 'timestamp_unix_ms',
           'pm2_5_ugm3', 'pm10_ugm3', 'nitrogen_dioxide_ugm3', 'ozone_ugm3',
           'sulphur_dioxide_ugm3', 'carbon_monoxide_ugm3', 'data_source',
           'ingestion_timestamp_utc', 'load_id', 'pipeline_version', 'record_hash']
for col in original:
    if col in df.columns:
        print(f"  • {col}")

print(f"\nTemporal features ({len([c for c in df.columns if any(x in c for x in ['year', 'month', 'day', 'hour', 'weekday', 'sin', 'cos', 'is_'])])})")
for col in df.columns:
    if any(x in col for x in ['year', 'month', 'day', 'hour', 'weekday', 'sin', 'cos', 'is_']):
        print(f"  • {col}")

print(f"\nLag features ({len([c for c in df.columns if 'lag' in c])})")
for col in df.columns:
    if 'lag' in col:
        print(f"  • {col}")

print(f"\nRolling features ({len([c for c in df.columns if 'rolling' in c])})")
for col in df.columns:
    if 'rolling' in col:
        print(f"  • {col}")

## Data Quality Check

In [ ]:
print("\n🔍 Data Quality Report:")
print("\nNull counts:")

key_cols = ['pm2_5_ugm3', 'pm10_ugm3', 'nitrogen_dioxide_ugm3', 'ozone_ugm3']
for col in key_cols:
    null_count = df[col].null_count()
    null_pct = (null_count / len(df)) * 100
    print(f"  {col:30s}: {null_count:8,} ({null_pct:5.2f}%)")

print("\nYearly coverage:")
yearly = df.group_by(pl.col('timestamp_utc').dt.year()).agg([
    pl.len().alias('rows'),
    pl.col('stationID').n_unique().alias('stations')
]).sort('timestamp_utc')

for row in yearly.iter_rows(named=True):
    year = row['timestamp_utc']
    rows = row['rows']
    stations = row['stations']
    expected = 365 * 24 * 79 if year != 2024 else 366 * 24 * 79
    coverage = (rows / expected) * 100
    print(f"  {year}: {rows:,} rows | {stations} stations | {coverage:.1f}% coverage")

## Sample Data

In [ ]:
print("\n📋 Sample rows (first 5):")
print(df.select(['stationID', 'timestamp_utc', 'pm2_5_ugm3', 'hour', 'weekday', 
                'pm2_5_ugm3_lag_1h', 'pm2_5_ugm3_rolling_mean_24h']).head(5))

## Quick Visualization

In [ ]:
# PM2.5 distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
pm25_values = df['pm2_5_ugm3'].to_numpy()
axes[0].hist(pm25_values, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('PM2.5 (µg/m³)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('PM2.5 Distribution (2023-2025)')
axes[0].axvline(df['pm2_5_ugm3'].mean(), color='red', linestyle='--', label=f"Mean: {df['pm2_5_ugm3'].mean():.1f}")
axes[0].legend()

# Box plot by year
df_pd = df.select(['timestamp_utc', 'pm2_5_ugm3']).to_pandas()
df_pd['year'] = df_pd['timestamp_utc'].dt.year
df_pd.boxplot(column='pm2_5_ugm3', by='year', ax=axes[1])
axes[1].set_xlabel('Year')
axes[1].set_ylabel('PM2.5 (µg/m³)')
axes[1].set_title('PM2.5 by Year')
plt.suptitle('')

plt.tight_layout()
plt.show()

print("\n✅ Visualization complete!")

## Next Steps

1. **Train ML Model**: ใช้ข้อมูล Gold layer นี้สำหรับ train โมเดล PM2.5 forecasting
2. **Feature Selection**: วิเคราะห์ว่า features ไหนมีความสำคัญ
3. **Time Series Split**: แบ่งข้อมูลเป็น train/val/test ตามลำดับเวลา
4. **Model Evaluation**: ประเมินผลโมเดลด้วย RMSE, MAE, R²